# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the FAIR² dataset package using the `mlcroissant` library in Python. We'll: 
* Load dataset metadata and records
* Review available record sets, fields, columns (by their `@id`)
* Load tables into pandas DataFrames
* Perform typical exploratory data analysis (EDA)
* Visualize trends and relationships

### Dataset Source
Croissant schema: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
We load the metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}")

## 2. Data Overview
Let's review the available record sets and the fields (columns, features) inside each one.
All dataset components are referred to by their Croissant `@id`.

In [ ]:
from pprint import pprint

# Helper to print all record sets, their IDs, and field (column) IDs
record_set_objs = []
if hasattr(metadata, 'recordSet'):
    if isinstance(metadata.recordSet, dict):
        record_set_objs = [metadata.recordSet]
    elif isinstance(metadata.recordSet, list):
        record_set_objs = metadata.recordSet
    else:
        record_set_objs = []

record_set_ids = []
print("Available Record Sets and their structure:")
for rs in record_set_objs:
    rs_id = getattr(rs, '@id', None) or getattr(rs, 'id', None)
    rs_name = getattr(rs, 'name', None)
    print(f"- RecordSet @id: {rs_id} | name: {rs_name}")
    record_set_ids.append(rs_id)
    # Show columns/fields for this RecordSet
    try:
        if hasattr(rs, 'field'):
            fields = rs.field
            if isinstance(fields, list):
                for field in fields:
                    f_id = getattr(field, '@id', None) or getattr(field, 'id', None)
                    f_name = getattr(field, 'name', None)
                    print(f"    - Field @id: {f_id} | name: {f_name}")
            elif isinstance(fields, dict):
                f_id = getattr(fields, '@id', None) or getattr(fields, 'id', None)
                f_name = getattr(fields, 'name', None)
                print(f"    - Field @id: {f_id} | name: {f_name}")
    except Exception as e:
        print("    Could not retrieve fields.")

if not record_set_ids:
    # If the schema uses the old 'hasPart' way (which wraps 'RecordSets'), try that
    if hasattr(metadata, 'hasPart') and metadata.hasPart:
        print('\nFalling back to hasPart lookup...\n')
        for part in metadata.hasPart:
            part_id = getattr(part, '@id', None) if hasattr(part, '@id') else None
            part_type = getattr(part, '@type', None) if hasattr(part, '@type') else None
            if part_type == 'cr:RecordSet':
                print(f"- RecordSet @id: {part_id}")
                record_set_ids.append(part_id)

if not record_set_ids:
    print('No RecordSets found in metadata. Double check the schema definition.')

### Optionally: Print a sample record from the first available record set
This helps to see the structure and available fields with real data.

In [ ]:
# Get the first RecordSet's @id for demonstration purposes
if not record_set_ids:
    print("No record sets found. Please check the dataset schema.")
else:
    first_rs_id = record_set_ids[0] if len(record_set_ids) > 0 else None
    print(f"First RecordSet for sampling: {first_rs_id}")
    try:
        rec_gen = dataset.records(record_set=first_rs_id)
        first_record = next(rec_gen)
        pprint(first_record)
    except Exception as e:
        print(f"Could not retrieve a record: {e}")

## 3. Data Extraction
We now load data from available record sets into pandas DataFrames for analysis. Use RecordSet and Field `@id`s from above for precise extraction and reproducibility.

In [ ]:
dfs = {}
for rs_id in record_set_ids:
    try:
        # Load all records for this record set as a DataFrame
        recs = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(recs)
        dfs[rs_id] = df
        print(f"Loaded {len(df)} records for RecordSet @id: {rs_id}")
    except Exception as e:
        print(f"Failed to load records for RecordSet {rs_id}: {e}")

if record_set_ids:
    example_rs_id = record_set_ids[0]
    print(f"Columns in DataFrame for {example_rs_id}:\n{dfs[example_rs_id].columns.tolist()}")
    display(dfs[example_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
We show how to process, filter, and analyze numeric fields from the data using their Croissant `@id`s.

- We'll choose a numeric field for the demonstration.
- Filter out records by threshold.
- Normalize that column.
- Optionally group or aggregate.

In [ ]:
# Pick a record set to analyze (the first one available)
record_set_to_use = record_set_ids[0] if len(record_set_ids) > 0 else None
if not record_set_to_use:
    raise ValueError('No record set available for EDA.')

df = dfs[record_set_to_use]

# Inspect columns to find a candidate numeric column
numeric_candidates = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
if not numeric_candidates:
    # Try loosening criteria: sometimes numeric columns are loaded as object due to missing values or parsing
    for col in df.columns:
        try:
            # Try converting to float
            df[col + '_float'] = pd.to_numeric(df[col], errors='coerce')
            if df[col + '_float'].notnull().sum() > 0:
                numeric_candidates.append(col)
        except:
            pass
    if numeric_candidates:
        # Use converted float columns
        df = df.rename(columns={col + '_float': col for col in numeric_candidates})

# Pick the first available numeric column
numeric_field = numeric_candidates[0] if numeric_candidates else None
if not numeric_field:
    print('No numeric field found for EDA demonstration.')
else:
    print(f"Using '{numeric_field}' as the numeric field for EDA.")

    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
    print(f"Filtering records with {numeric_field} > {threshold:.2f}\n")
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered sample records:")
    display(filtered_df.head())

    # Normalize the numeric field z-score style
    field_mean = filtered_df[numeric_field].mean()
    field_std = filtered_df[numeric_field].std()
    filtered_df[f'{numeric_field}_normalized'] = (filtered_df[numeric_field] - field_mean) / (field_std if field_std != 0 else 1)
    print(f"Normalized '{numeric_field}' for filtered records:")
    display(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())

    # Check for a categorical/groupable field
    # Suggest first non-numeric field as a group
    non_numeric_candidates = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])][:5]
    group_field = non_numeric_candidates[0] if non_numeric_candidates else None
    if group_field is not None and group_field in filtered_df.columns:
        print(f"Grouping by '{group_field}' and aggregating mean of normalized {numeric_field}...")
        grouped_df = filtered_df.groupby(group_field)[f"{numeric_field}_normalized"].mean()
        display(grouped_df.head())
    else:
        print("No obvious categorical field for grouping.")

## 5. Visualization
Let's visualize the distribution of the chosen numeric variable and its normalized version, and see any interesting groupings if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    fig, axs = plt.subplots(1, 2, figsize=(12,5))
    sns.histplot(df[numeric_field].dropna(), kde=True, ax=axs[0])
    axs[0].set_title(f"Distribution of '{numeric_field}'")
    if f'{numeric_field}_normalized' in filtered_df.columns:
        sns.histplot(filtered_df[f'{numeric_field}_normalized'].dropna(), kde=True, ax=axs[1], color='orange')
        axs[1].set_title(f"'{numeric_field}' Normalized (filtered)")
    plt.tight_layout()
    plt.show()

    # Optionally boxplot per group if grouped_df is available
    if 'group_field' in locals() and group_field is not None and group_field in filtered_df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
        plt.title(f"'{numeric_field}' by {group_field}")
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² dataset using the `mlcroissant` library. We demonstrated how to:
- Load Croissant metadata and discover available tables (`RecordSet`) and features (`@id`)
- Extract data into pandas DataFrames
- Perform numeric filtering, normalization, and simple group-based aggregation
- Visualize data distributions

This workflow can be adapted for any Croissant-standard dataset for reproducible, FAIR-compliant data science.